# Exp_18: Design, Train and Test a recurrent neural network (RNN) on a dataset to predict the next number in the following sequence X = [1,4,7,11,14].

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [2]:
# Define the RNN model
class RNNPredictor(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=1):
        super(RNNPredictor, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x, h):
        out, h = self.rnn(x, h)
        out = self.fc(out[:, -1, :])  # Get last time step output
        return out, h

In [3]:
# Generate data
def create_data(sequence):
    X, Y = [], []
    for i in range(len(sequence) - 1):
        X.append([sequence[i]]) # 1, Hel, hi
        Y.append(sequence[i+1]) # 4, GM
    return torch.tensor(X, dtype=torch.float32).unsqueeze(1), 
    torch.tensor(Y, dtype=torch.float32)

In [4]:
# Define sequence and data
sequence = [1, 4, 7, 11, 14, 17, 20]
X_train, Y_train = create_data(sequence)

In [5]:
X_train, Y_train

(tensor([[[ 1.]],
 
         [[ 4.]],
 
         [[ 7.]],
 
         [[11.]],
 
         [[14.]],
 
         [[17.]]]),
 tensor([ 4.,  7., 11., 14., 17., 20.]))

In [6]:
# Model parameters
input_size = 1
hidden_size = 16
output_size = 1
num_layers = 1

In [7]:
# Initialize model, loss, and optimizer
model = RNNPredictor(input_size, hidden_size, output_size, num_layers)
criterion = nn.MSELoss() # MSE - Mean Square Error Loss
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [8]:
# Training loop
epochs = 1000
for epoch in range(epochs):
    batch_size = X_train.shape[0]  # Get batch size dynamically
    h = torch.zeros(num_layers, batch_size, hidden_size)  # Initialize hidden state
    
    optimizer.zero_grad()
    output, _ = model(X_train, h)
    loss = criterion(output.squeeze(), Y_train)
    loss.backward() # Backpropagation through time
    optimizer.step()

    if (epoch + 1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [20/1000], Loss: 102.8685
Epoch [40/1000], Loss: 56.1150
Epoch [60/1000], Loss: 32.3730
Epoch [80/1000], Loss: 19.4885
Epoch [100/1000], Loss: 11.9735
Epoch [120/1000], Loss: 7.7451
Epoch [140/1000], Loss: 5.2559
Epoch [160/1000], Loss: 3.7004
Epoch [180/1000], Loss: 2.6745
Epoch [200/1000], Loss: 1.9707
Epoch [220/1000], Loss: 1.4838
Epoch [240/1000], Loss: 1.1515
Epoch [260/1000], Loss: 0.9221
Epoch [280/1000], Loss: 0.7565
Epoch [300/1000], Loss: 0.6313
Epoch [320/1000], Loss: 0.5332
Epoch [340/1000], Loss: 0.4544
Epoch [360/1000], Loss: 0.3899
Epoch [380/1000], Loss: 0.3360
Epoch [400/1000], Loss: 0.2897
Epoch [420/1000], Loss: 0.2487
Epoch [440/1000], Loss: 0.2109
Epoch [460/1000], Loss: 0.1733
Epoch [480/1000], Loss: 0.1384
Epoch [500/1000], Loss: 0.1111
Epoch [520/1000], Loss: 0.0887
Epoch [540/1000], Loss: 0.0703
Epoch [560/1000], Loss: 0.0554
Epoch [580/1000], Loss: 0.0435
Epoch [600/1000], Loss: 0.0340
Epoch [620/1000], Loss: 0.0266
Epoch [640/1000], Loss: 0.0208
Epoch 

In [11]:
# Testing the model
h = torch.zeros(num_layers, 1, hidden_size)  # Batch size = 1 for testing
x_test = torch.tensor([[14]], dtype=torch.float32).unsqueeze(0)  # Shape: (1, 1, 1)
predicted, _ = model(x_test, h)
print(f'Predicted next number: {predicted.item():.2f}')

Predicted next number: 17.02


# Task: Train the model with [1:1000] odd and even numbers.
## Obj: If input is odd it should predict odd, else even

# What are my eq's in RNN model h(t) = tanh(h(t-1)*Wh + x(t))*Wx + bh), Y = sigma(h(t)*wy + by)

In [12]:
import numpy as np

In [13]:
class SimpleRNNCell:
    def __init__(self, input_size, hidden_size, output_size):
        self.hidden_size = hidden_size

        # Weight initialization
        self.W_x = np.random.randn(hidden_size, input_size)*0.1
        self.W_h = np.random.randn(hidden_size, hidden_size)*0.1
        self.b_h = np.zeros((hidden_size,1))
        self.W_y = np.random.randn(output_size, hidden_size) * 0.1
        self.b_y = np.zeros((output_size, 1))

    def forward(self, x_t, h_prev):
        # Compute new hidden state
        h_t = np.tanh(np.dot(self.W_x, x_t) + np.dot(self.W_h, h_prev) + self.b_h)
        
        # Compute output
        y_t = np.dot(self.W_y, h_t) + self.b_y
        
        return h_t, y_t

In [14]:
# Training data
#sequence = np.array([1, 4, 7, 11, 14])
sequence = np.array([1, 2, 3,4,5])
X_train = sequence[:-1].reshape(-1, 1, 1)  # Inputs: [1], [4], [7], [11]
Y_train = sequence[1:].reshape(-1, 1, 1)   # Targets: [4], [7], [11], [14]

In [15]:
# Hyperparameters
input_size = 1
hidden_size = 5
output_size = 1
learning_rate = 0.01
epochs = 500

In [16]:
# Initialize RNN cell
rnn_cell = SimpleRNNCell(input_size, hidden_size, output_size)

In [17]:
# Training loop
h_prev = np.zeros((hidden_size, 1))  # Initial hidden state
for epoch in range(epochs):
    total_loss = 0
    
    for i in range(len(X_train)):
        x_t = X_train[i].reshape(1, 1)
        y_true = Y_train[i].reshape(1, 1)
        
        # Forward pass
        h_t, y_pred = rnn_cell.forward(x_t, h_prev)
        
        # Compute loss (Mean Squared Error)
        loss = np.square(y_true - y_pred).sum()
        total_loss += loss
        
        # Backpropagation (Gradient Descent)
        dL_dy = -2 * (y_true - y_pred)  # Gradient of loss w.r.t output
        dL_dWy = np.dot(dL_dy, h_t.T)
        dL_dby = dL_dy
        
        dL_dh = np.dot(rnn_cell.W_y.T, dL_dy) * (1 - h_t**2)  # tanh derivative
        dL_dWx = np.dot(dL_dh, x_t.T)
        dL_dWh = np.dot(dL_dh, h_prev.T)
        dL_dbh = dL_dh
        
        # Update weights
        rnn_cell.W_y -= learning_rate * dL_dWy
        rnn_cell.b_y -= learning_rate * dL_dby
        rnn_cell.W_x -= learning_rate * dL_dWx
        rnn_cell.W_h -= learning_rate * dL_dWh
        rnn_cell.b_h -= learning_rate * dL_dbh
        
        h_prev = h_t  # Update hidden state for next time step
    
    if (epoch + 1) % 50 == 0:
        print(f'Epoch {epoch+1}, Loss: {total_loss:.4f}')

Epoch 50, Loss: 0.3490
Epoch 100, Loss: 0.0861
Epoch 150, Loss: 0.0492
Epoch 200, Loss: 0.0413
Epoch 250, Loss: 0.0349
Epoch 300, Loss: 0.0290
Epoch 350, Loss: 0.0239
Epoch 400, Loss: 0.0194
Epoch 450, Loss: 0.0156
Epoch 500, Loss: 0.0124


In [18]:
# Testing the RNN
h_prev = np.zeros((hidden_size, 1))
x_test = np.array([[5]])  # Predict next number
h_t, y_pred = rnn_cell.forward(x_test, h_prev)
print(f'Predicted next number: {y_pred.item():.2f}')

Predicted next number: 5.39


In [19]:
# Calculate total trainable parameters
def count_parameters():
    param_count = (
        rnn_cell.W_x.size +  # Input weight
        rnn_cell.W_h.size +  # Hidden weight
        rnn_cell.b_h.size +  # Hidden bias
        rnn_cell.W_y.size +  # Output weight
        rnn_cell.b_y.size    # Output bias
    )
    return param_count

print(f'Total trainable parameters: {count_parameters()}')

Total trainable parameters: 41


In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Create a directed graph
G = nx.DiGraph()

# Add nodes
G.add_node("x_t", color="blue")
G.add_node("h_t-1", color="orange")
G.add_node("h_t", color="green")
G.add_node("y_t", color="red")
G.add_node("W_x", color="gray")
G.add_node("W_h", color="gray")
G.add_node("W_y", color="gray")
G.add_node("b_h", color="gray")
G.add_node("b_y", color="gray")
G.add_node("tanh", color="purple")

# Add edges
edges = [
    ("x_t", "W_x"), ("W_x", "h_t"),
    ("h_t-1", "W_h"), ("W_h", "h_t"),
    ("b_h", "h_t"),
    ("h_t", "tanh"), ("tanh", "h_t"),
    ("h_t", "W_y"), ("W_y", "y_t"),
    ("b_y", "y_t")
]

G.add_edges_from(edges)

# Define positions
pos = {
    "x_t": (-1, 1), "W_x": (0, 1), "h_t-1": (-1, 0), "W_h": (0, 0),
    "h_t": (1, 0.5), "tanh": (0.5, 0.5), "W_y": (1.5, 0.5), "y_t": (2, 0.5),
    "b_h": (0, -0.5), "b_y": (1.5, -0.5)
}

# Draw the graph
plt.figure(figsize=(8, 6))
colors = [G.nodes[n]["color"] for n in G.nodes]
nx.draw(G, pos, with_labels=True, node_color=colors, node_size=3000, edge_color="black", font_size=10)
plt.title("Simple RNN Cell Visualization")
plt.show()

# Hidden States = 5, h_t = 5, How many inputs for 5 hidden states, = 5 with its own weight. W_x = 5x1 = 5
# W_h = 5(h_t) X 5(h_t-1) = 25
# bh = 5
# W_y = 5 X 1 = 5
# b_y = 1
# Total Trainable Parameters = 41

# 1. You have to show how RNN model works on Long sequence data. 20 to 30 samples.
# 2. Prove vanising gradients in RNN model.